# `05_bicycle_route_infrastructure_per_side`: Unnesting and Protection Classification

## Introduction

### Purpose

This notebook derives `bicycle_route_infrastructure_per_side` from `bicycle_route_m_ways_distinct_classified`. It is the only step in the pipeline where the grain changes: from one row per way (181,318 ways) to one row per way per carriageway side (198,540 side-rows, per thesis §3.4.2). Each side-row is then assigned a protection level, a UNECE infrastructure class, an evidence basis, and a classifiability outcome, completing the per-side classification framework the thesis develops in §3.4.3 and §3.4.4. Total network length at the per-side grain is 41,814.7 km (thesis §3.4.2).

> **Facility length, not road length.** Each row in `bicycle_route_infrastructure_per_side` carries `way_m_length_nl_meters`, the full length of the original way. For a carriageway way the left and right rows carry the same value, so summing across both sides double-counts road length. This is intentional: a 500 m road with cycle tracks on both sides provides 1,000 m of facility, the cyclist-centric quantity the thesis seeks (§3.4.2). Aggregations that want road length instead must divide carriageway km by 2 or use `COUNT(DISTINCT way_m_id)` as the base.

### Inputs

- `bicycle_route_m_ways_distinct_classified` from `04_bicycle_route_m_ways_distinct_classified` (181,318 ways with the full Step-1-to-Step-9 column set, including `mapping_style`, `effective_cycleway_left/right`, `infrastructure_confidence`).

### Outputs

- `bicycle_route_infrastructure_per_side`: one row per way per carriageway side (198,540 rows), carrying all columns from `bicycle_route_m_ways_distinct_classified` plus:
  - `side`: `left` / `right` / `dedicated` / `n/a` / `ferry` / `tagging_conflict`.
  - `cycleway_side`: `effective_cycleway_left` or `effective_cycleway_right` for carriageway sides; `NULL` otherwise.
  - `protection_level_per_side`: `protected` / `unprotected` / `no_infrastructure` / `separate_infrastructure` / `mixed_traffic_uncertain` / `uncertain` / `ferry` / `tagging_conflict`.
  - `unece_class`: `cycle_track` / `cycle_lane` / `cycle_street` / `separate` / `mixed_traffic` / `roundabout` / `tagging_conflict` / `ferry` / `other` (thesis Table 1).
  - `evidence_basis`: `recorded_presence` / `recorded_absence_explicit` / `recorded_absence_sidepath` / `inferred_absence_partial` / `inferred_absence_none` / `excluded` (thesis §3.4.3, §3.4.4).
  - `classifiability`: `classifiable` / `unclassifiable` / `excluded` (thesis §3.4.3).

### Key steps

The unnesting and protection classification happen in a single query because protection classification reads `side` and `cycleway_side`, which only exist after unnesting. The query first unnests via `UNION ALL`: `dedicated_mapping`, `n/a`, `ferry`, and `tagging_conflict` rows are emitted once each with the corresponding `side` label, while `carriageway_mapping` rows are emitted twice (once with `side='left'` and `cycleway_side = effective_cycleway_left`, once with `side='right'` and `cycleway_side = effective_cycleway_right`). A `CASE` then assigns `protection_level_per_side`. A second query on top assigns `unece_class`, `evidence_basis`, and `classifiability`. Only carriageway sides at `infrastructure_confidence = 'certain'` receive a definitive protection classification; all others are conservatively marked `uncertain`, which the thesis frames as the inference vs. recorded distinction that prevents the headline figures from inheriting certainty the data does not contain (§3.4.3).

### Dependencies on prior notebooks

- `04_bicycle_route_m_ways_distinct_classified.ipynb`: provides `bicycle_route_m_ways_distinct_classified` (and transitively brings in `00_bicycle_route_relations` and `03_boundaries_population`).

### Downstream consumers

- `06_bicycle_route_infrastructure_per_side_per_spatial_unit`: clips this table to municipality / province / H3 boundaries and computes `clipped_length_meters` per side per spatial unit (stage 06 spatial join, thesis §3.4.6).
- `07_bicycle_route_infrastructure_per_side_metrics`: aggregates facility length and the classifiability metrics by spatial unit. This is the table the thesis headline results draw from (classifiable share, evidence-basis decomposition, urban–rural gradient, LISA clustering, thesis Results).

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Grain change: from way to side](#2-grain-change-from-way-to-side)
3. [Result: `bicycle_route_infrastructure_per_side`](#3-result-bicycle_route_infrastructure_per_side)
4. [Validation](#4-validation)
   - 4.1 [Row count](#41-row-count)
   - 4.2 [Mapping-style consistency before and after](#42-mapping-style-consistency-before-and-after)
   - 4.3 [Distribution of protection levels](#43-distribution-of-protection-levels)
   - 4.4 [Check for unhandled rows](#44-check-for-unhandled-rows)
5. [Downstream use](#5-downstream-use)

---

## 1. Environment setup

### Libraries and extensions

In [2]:
import duckdb
from IPython.utils import io

### Loading shared variables

Execute notebook 04 to bring `bicycle_route_m_ways_distinct_classified` into the current session.

In [3]:
with io.capture_output() as captured:
    %run /home/vbo226/04_bicycle_route_m_ways_distinct_classified.ipynb

This notebook adds no new enrichment columns beyond what notebook 04 produced; it only changes the grain and assigns per-side protection, UNECE class, evidence basis, and classifiability.

---

## 2. Grain change: from way to side

**Why per side.** A single way can have a protected cycle track on one side and an unprotected lane on the other. A way-level label would either overstate or understate the protection available. The per-side representation preserves this asymmetry, which the thesis identifies (§3.4.2) as essential for computing facility length accurately and for distinguishing partial-survey patterns (one side tagged, the other null) from no-survey patterns (both sides null).

**How the unnesting works.** `mapping_style` determines how many rows each way produces and what value `cycleway_side` carries:

| `mapping_style` | Rows produced | `side` value | `cycleway_side` value |
|---|---|---|---|
| `carriageway_mapping` | 2 | `'right'` and `'left'` | `effective_cycleway_right` and `effective_cycleway_left` respectively |
| `dedicated_mapping` | 1 | `'dedicated'` | `NULL` |
| `n/a` | 1 | `'n/a'` | `NULL` |
| `ferry` | 1 | `'ferry'` | `NULL` |
| `tagging_conflict` | 1 | `'tagging_conflict'` | `NULL` |

**Protection level assignment.** Only carriageway sides at `infrastructure_confidence = 'certain'` receive a definitive protection classification; all other carriageway sides are classified as `uncertain`. This is conservative by design: assigning protection levels where tagging is incomplete would misrepresent what can be concluded from the data (thesis §3.4.3).

| `protection_level_per_side` | Condition |
|---|---|
| `protected` | `dedicated_mapping`, or carriageway `cycleway_side = 'track'` at `certain` confidence |
| `unprotected` | Carriageway `cycleway_side IN ('lane', 'shared_lane', 'share_busway')` at `certain` confidence |
| `no_infrastructure` | Carriageway `cycleway_side = 'no'` or `NULL` at `certain` confidence |
| `separate_infrastructure` | Carriageway `cycleway_side = 'separate'` at `certain` confidence |
| `mixed_traffic_uncertain` | `mapping_style = 'n/a'`; cyclist likely uses mixed traffic but not confirmed |
| `uncertain` | Carriageway side where `infrastructure_confidence != 'certain'` |
| `ferry` | Ferry route |
| `tagging_conflict` | Conflicting tags; excluded from analysis |

## 3. Result: `bicycle_route_infrastructure_per_side`

The result is produced in two queries layered on the unnested table. The first (`bicycle_route_infrastructure_per_side_with_protection_level`) does the `UNION ALL` unnesting and assigns `protection_level_per_side`. The second (`bicycle_route_infrastructure_per_side`) maps protection levels and contextual tags onto the thesis's three classification columns: `unece_class` (Table 1: OSM tags → UNECE infrastructure class), `evidence_basis` (§3.4.3 / §3.4.4: recorded vs. inferred presence/absence), and `classifiability` (§3.4.3: collapse of evidence basis into classifiable / unclassifiable / excluded, the column the headline 52.2% figure aggregates from).

**Evidence-basis mapping.**

| `evidence_basis` | Source condition |
|---|---|
| `recorded_presence` | `unece_class IN ('cycle_track', 'cycle_lane', 'cycle_street')`: infrastructure type is positively tagged |
| `recorded_absence_explicit` | `unece_class = 'mixed_traffic'` AND `infrastructure_confidence = 'certain'`: `cycleway=no` confirmed |
| `recorded_absence_sidepath` | `unece_class = 'mixed_traffic'` AND `infrastructure_confidence IN ('high_explained_by_sidepath', 'high_use_sidepath_confirms_no_infrastructure')`: absence stated via `use_sidepath` |
| `inferred_absence_partial` | `unece_class = 'mixed_traffic'` AND `infrastructure_confidence = 'medium_side_null_unexplained'`: one side tagged, the other null (thesis §3.4.4) |
| `inferred_absence_none` | `unece_class = 'mixed_traffic'` AND `infrastructure_confidence = 'low_no_cycleway_infrastructure_signal'`: no cycling signal at all (thesis §3.4.4) |
| `excluded` | Roundabouts, ferries, tagging conflicts, separate-mapped, "other" |

`classifiability` then folds these into three outcomes: `classifiable` for any `recorded_*`, `unclassifiable` for any `inferred_*`, and `excluded` for the rest.

In [4]:
bicycle_route_infrastructure_per_side_with_protection_level = duckdb.sql("""
WITH unnested AS (
    SELECT *,
        'dedicated'        AS side, NULL                    AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'dedicated_mapping'
    UNION ALL
    SELECT *,
        'tagging_conflict' AS side, NULL                    AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'tagging_conflict'
    UNION ALL
    SELECT *,
        'ferry'            AS side, NULL                    AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'ferry'
    UNION ALL
    SELECT *,
        'n/a'              AS side, NULL                    AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'n/a'
    UNION ALL
    SELECT *,
        'left'             AS side, effective_cycleway_left  AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'carriageway_mapping'
    UNION ALL
    SELECT *,
        'right'            AS side, effective_cycleway_right AS cycleway_side
    FROM bicycle_route_m_ways_distinct_classified WHERE mapping_style = 'carriageway_mapping'
)
SELECT *,
    CASE
        WHEN side = 'dedicated'
            THEN 'protected'

        WHEN side IN ('n/a')
            THEN 'mixed_traffic_uncertain'

        WHEN side IN ('ferry')
            THEN 'ferry'

        WHEN side IN ('tagging_conflict')
            THEN 'tagging_conflict'

        -- carriageway: only classify certain confidence
        WHEN side IN ('left', 'right')
         AND infrastructure_confidence != 'certain'
            THEN 'uncertain'

        WHEN side IN ('left', 'right')
         AND cycleway_side = 'track'
            THEN 'protected'

        WHEN side IN ('left', 'right')
         AND cycleway_side IN ('lane', 'shared_lane', 'share_busway')
            THEN 'unprotected'

        WHEN side IN ('left', 'right')
         AND cycleway_side = 'no'
            THEN 'no_infrastructure'

        WHEN side IN ('left', 'right')
         AND cycleway_side = 'separate'
            THEN 'separate_infrastructure'

        WHEN side IN ('left', 'right')
         AND cycleway_side IS NULL
            THEN 'no_infrastructure'

        ELSE 'unhandled'
    END AS protection_level_per_side

FROM unnested
""")

In [5]:
bicycle_route_infrastructure_per_side = duckdb.sql("""
WITH typed AS (
  SELECT
    s.*,
    CASE
      WHEN s.protection_level_per_side = 'protected'                    THEN 'cycle_track'
      WHEN way_m_tags['cyclestreet'] = 'yes'
        OR way_m_tags['bicycle_road'] = 'yes'                           THEN 'cycle_street'
      WHEN s.protection_level_per_side = 'unprotected'                  THEN 'cycle_lane'
      WHEN s.protection_level_per_side = 'separate_infrastructure'      THEN 'separate'
      WHEN s.infrastructure_confidence = 'high_explained_by_roundabout' THEN 'roundabout'
      WHEN s.infrastructure_confidence = 'tagging_conflict'             THEN 'tagging_conflict'
      WHEN s.infrastructure_confidence = 'ferry'                        THEN 'ferry'
      WHEN s.protection_level_per_side IN
           ('no_infrastructure','mixed_traffic_uncertain','uncertain')  THEN 'mixed_traffic'
      ELSE 'other'
    END AS unece_class
  FROM bicycle_route_infrastructure_per_side_with_protection_level s
),
based AS (
  SELECT
    t.*,
    CASE
      -- infrastructure type is positively tagged
      WHEN unece_class IN ('cycle_track','cycle_lane','cycle_street')
           THEN 'recorded_presence'
      -- absence stated explicitly: cycleway=no
      WHEN unece_class = 'mixed_traffic' AND infrastructure_confidence = 'certain'
           THEN 'recorded_absence_explicit'
      -- absence stated via use_sidepath directive
      -- (surveyed + unsurveyed collapse here; raw infrastructure_confidence is kept via t.* for the split)
      WHEN unece_class = 'mixed_traffic' AND infrastructure_confidence IN
           ('high_explained_by_sidepath','high_use_sidepath_confirms_no_infrastructure')
           THEN 'recorded_absence_sidepath'
      -- absence inferred from a surveyed one-sided null
      WHEN unece_class = 'mixed_traffic' AND infrastructure_confidence = 'medium_side_null_unexplained'
           THEN 'inferred_absence_partial'
      -- absence inferred from no cycling signal at all
      WHEN unece_class = 'mixed_traffic' AND infrastructure_confidence = 'low_no_cycleway_infrastructure_signal'
           THEN 'inferred_absence_none'
      -- special cases removed from the split
      ELSE 'excluded'
    END AS evidence_basis
  FROM typed t
)
SELECT
  *,
  CASE
    WHEN evidence_basis LIKE 'recorded_%'  THEN 'classifiable'
    WHEN evidence_basis LIKE 'inferred_%'  THEN 'unclassifiable'
    ELSE 'excluded'
  END AS classifiability
FROM based;
""")

---

## 4. Validation

### 4.1 Row count

The total number of side-rows must equal `n_non_carriageway × 1 + n_carriageway × 2`. The thesis reports 198,540 side-rows for 181,318 input ways (§3.4.2); the check below recomputes this from scratch.

In [6]:
n_ways  = duckdb.sql("SELECT COUNT(*) FROM bicycle_route_m_ways_distinct_classified").fetchone()[0]
n_sides = duckdb.sql("SELECT COUNT(*) FROM bicycle_route_infrastructure_per_side").fetchone()[0]
 
n_carriageway = duckdb.sql("""
    SELECT COUNT(*) FROM bicycle_route_m_ways_distinct_classified
    WHERE mapping_style = 'carriageway_mapping'
""").fetchone()[0]
 
n_non_carriageway = n_ways - n_carriageway
 
expected_sides = n_non_carriageway + (n_carriageway * 2)
 
print(f"Input ways (total):          {n_ways:>10,}")
print(f"  of which carriageway:      {n_carriageway:>10,}")
print(f"  of which non-carriageway:  {n_non_carriageway:>10,}")
print(f"Expected side-rows:          {expected_sides:>10,}  (non-carriageway × 1 + carriageway × 2)")
print(f"Actual side-rows:            {n_sides:>10,}")
print(f"Match:                       {n_sides == expected_sides}")
print(f"Ratio (sides/ways):          {n_sides/n_ways:>10.2f}x")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Input ways (total):             181,318
  of which carriageway:          17,222
  of which non-carriageway:     164,096
Expected side-rows:             198,540  (non-carriageway × 1 + carriageway × 2)
Actual side-rows:               198,540
Match:                       True
Ratio (sides/ways):                1.09x


### Mapping style consistency before and### 4.2 Mapping-style consistency before and after

`n_ways` and `road_km` should match exactly between before and after unnesting, for every `mapping_style`. A mismatch in `n_ways` indicates rows were dropped or duplicated. A mismatch in `road_km` indicates a length-computation error. Carriageway km is divided by 2 after unnesting to recover road km from facility km, per the convention established in the Introduction. after unnesting

In [7]:
# Before unnesting
before = duckdb.sql("""
SELECT
    mapping_style,
    COUNT(*)                                        AS n_ways,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 1)       AS road_km
FROM bicycle_route_m_ways_distinct_classified
GROUP BY 1
ORDER BY n_ways DESC
""").df()
 
# After unnesting, reconstruct mapping_style from side, divide carriageway km by 2
after = duckdb.sql("""
WITH labelled AS (
    SELECT *,
        CASE
            WHEN side IN ('left', 'right') THEN 'carriageway_mapping'
            ELSE side
        END AS mapping_style_label
    FROM bicycle_route_infrastructure_per_side
)
SELECT
    mapping_style_label                                         AS mapping_style,
    COUNT(DISTINCT way_m_id)                                    AS n_ways,
    ROUND(SUM(way_m_length_nl_meters) /
        CASE WHEN mapping_style_label = 'carriageway_mapping'
             THEN 2000.0 ELSE 1000.0 END, 1)                   AS road_km
FROM labelled
GROUP BY mapping_style_label
ORDER BY n_ways DESC
""").df()
 
print("Before unnesting:")
display(before)
print("After unnesting (carriageway km divided by 2 to recover road km):")
display(after)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Before unnesting:


,mapping_style,n_ways,road_km
0,n/a,88254,19827.3
1,dedicated_mapping,75646,15303.8
2,carriageway_mapping,17222,3243.8
3,ferry,186,195.1
4,tagging_conflict,10,0.8


After unnesting (carriageway km divided by 2 to recover road km):


,mapping_style,n_ways,road_km
0,n/a,88254,19827.3
1,dedicated,75646,15303.8
2,carriageway_mapping,17222,3243.8
3,ferry,186,195.1
4,tagging_conflict,10,0.8


`n_ways` and `road_km` should match exactly between before and after for every `mapping_style`. A mismatch in `n_ways` indicates rows were dropped or duplicated. A mismatch in `road_km` indicates a length computation error.

### 4.3 Distribution of protection levels

Headline distribution of `protection_level_per_side` across the 198,540 side-rows, and the same distribution broken down by `mapping_style` so the contribution of each mapping style to each protection level is visible.

In [8]:
duckdb.sql("""
SELECT
    protection_level_per_side,
    COUNT(*)                                                AS n_sides,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 1)              AS facility_km,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)     AS pct_sides
FROM bicycle_route_infrastructure_per_side
GROUP BY 1
ORDER BY n_sides DESC
""").show(max_width=150)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────┬─────────┬─────────────┬───────────┐
│ protection_level_per_side │ n_sides │ facility_km │ pct_sides │
│          varchar          │  int64  │   double    │  double   │
├───────────────────────────┼─────────┼─────────────┼───────────┤
│ mixed_traffic_uncertain   │   88254 │     19827.3 │     44.45 │
│ protected                 │   76002 │     15356.2 │     38.28 │
│ unprotected               │   18492 │      3754.7 │      9.31 │
│ no_infrastructure         │   10619 │      2388.5 │      5.35 │
│ uncertain                 │    4914 │       286.5 │      2.48 │
│ ferry                     │     186 │       195.1 │      0.09 │
│ separate_infrastructure   │      63 │         5.5 │      0.03 │
│ tagging_conflict          │      10 │         0.8 │      0.01 │
└───────────────────────────┴─────────┴─────────────┴───────────┘



In [9]:
# Breakdown by mapping_style within each protection level
duckdb.sql("""
SELECT
    mapping_style,
    protection_level_per_side,
    COUNT(*)                                                AS n_sides,
    ROUND(SUM(way_m_length_nl_meters) / 1000, 1)              AS facility_km
FROM bicycle_route_infrastructure_per_side
GROUP BY 1, 2
ORDER BY mapping_style, n_sides DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬───────────────────────────┬─────────┬─────────────┐
│    mapping_style    │ protection_level_per_side │ n_sides │ facility_km │
│       varchar       │          varchar          │  int64  │   double    │
├─────────────────────┼───────────────────────────┼─────────┼─────────────┤
│ carriageway_mapping │ unprotected               │   18492 │      3754.7 │
│ carriageway_mapping │ no_infrastructure         │   10619 │      2388.5 │
│ carriageway_mapping │ uncertain                 │    4914 │       286.5 │
│ carriageway_mapping │ protected                 │     356 │        52.4 │
│ carriageway_mapping │ separate_infrastructure   │      63 │         5.5 │
│ dedicated_mapping   │ protected                 │   75646 │     15303.8 │
│ ferry               │ ferry                     │     186 │       195.1 │
│ n/a                 │ mixed_traffic_uncertain   │   88254 │     19827.3 │
│ tagging_conflict    │ tagging_conflict          │      10 │         0.8 │
└───────────

### 4.4 Check for unhandled rows

`protection_level_per_side = 'unhandled'` would indicate a gap in the `CASE` logic of the unnesting query. The check below must return 0.

In [10]:
n_unhandled = duckdb.sql("""
    SELECT COUNT(*) FROM bicycle_route_infrastructure_per_side
    WHERE protection_level_per_side = 'unhandled'
""").fetchone()[0]
 
print(f"Unhandled rows: {n_unhandled:,}")
# Should be 0. Any unhandled row indicates a gap in the CASE logic

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unhandled rows: 0


---

## 5. Downstream use

`bicycle_route_infrastructure_per_side` feeds two downstream notebooks:

- **`06_bicycle_route_infrastructure_per_side_per_spatial_unit`** (stage 06 spatial join, thesis §3.4.6): clips this table to municipality, province, and H3 boundaries and computes `clipped_length_meters` per side per spatial unit. Produces `bicycle_route_infrastructure_per_side_per_municipality`, `_per_province`, and `_per_h3`.
- **`07_bicycle_route_infrastructure_per_side_metrics`** (stage 07 metrics): aggregates facility length by `protection_level_per_side`, `unece_class`, `evidence_basis`, and `classifiability` per spatial unit. This is the table the thesis headline results draw from: classifiable share (52.2%), unclassifiable share (47.3%), urban–rural gradient, evidence-basis decomposition (Figure 9, Figure 10), and LISA clustering.

Note that `way_m_length_nl_meters` in this table is the full unclipped way length. The clipped length per spatial unit, the correct denominator for facility metrics, is computed in `06_bicycle_route_infrastructure_per_side_per_spatial_unit` using the same geometric intersection approach as `06_bicycle_route_m_ways_distinct_classified_per_spatial_unit`.

In [12]:
duckdb.sql("""
SELECT unece_class, evidence_basis, classifiability, COUNT(*) as n_ways, 
    ROUND(SUM(length_nl_meters) / 1000.0, 3) as total_length
FROM bicycle_route_infrastructure_per_side
GROUP BY unece_class, evidence_basis, classifiability
ORDER BY total_length DESC
""").show(max_width=150)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬───────────────────────────┬─────────────────┬────────┬──────────────┐
│   unece_class    │      evidence_basis       │ classifiability │ n_ways │ total_length │
│     varchar      │          varchar          │     varchar     │ int64  │    double    │
├──────────────────┼───────────────────────────┼─────────────────┼────────┼──────────────┤
│ mixed_traffic    │ inferred_absence_none     │ unclassifiable  │  86501 │    19523.746 │
│ cycle_track      │ recorded_presence         │ classifiable    │  76002 │    15356.206 │
│ cycle_lane       │ recorded_presence         │ classifiable    │  18368 │     3730.233 │
│ mixed_traffic    │ recorded_absence_explicit │ classifiable    │  10579 │      2379.54 │
│ cycle_street     │ recorded_presence         │ classifiable    │   1763 │      332.032 │
│ mixed_traffic    │ inferred_absence_partial  │ unclassifiable  │   3292 │      243.166 │
│ ferry            │ excluded                  │ excluded        │    186 │      195.129 │

### Export

Persist `bicycle_route_infrastructure_per_side` to disk so the stage-06 spatial-join notebook and the stage-07 metrics notebook can read it without re-running the unnesting. `safe_write_parquet` matches the atomic-rename pattern used in `04_bicycle_route_m_ways_distinct_classified`.

In [51]:
from pathlib import Path
import os

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)


def safe_write_parquet(df, final_path):
    tmp_path = final_path + ".tmp"

    df.write_parquet(tmp_path)
    os.replace(tmp_path, final_path)

safe_write_parquet(
    bicycle_route_infrastructure_per_side,
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side.parquet"
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))